In [1]:
from pathlib import Path
import json

sections_dir = Path("temporary_sections")
responses_dir = Path("../../data/test/raw_api_responses")

audio_files = sorted(sections_dir.glob("*.wav"))

completed = {
    file.stem
    for file in responses_dir.glob("section_*.response")
}

pending = [
    file for file in audio_files
    if file.stem not in completed
]

print("Total audio sections:", len(audio_files))
print("Already transcribed:", len(audio_files) - len(pending))
print("Still need transcription:", len(pending))

print("\nFirst 5 pending sections:")
for file in pending[:5]:
    print(file.name)

Total audio sections: 10
Already transcribed: 10
Still need transcription: 0

First 5 pending sections:


In [2]:
import subprocess

original_audio = Path("../../data/test/test_audio.mp3")

def get_duration(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path)
        ],
        capture_output=True,
        text=True,
        check=True
    )

    return float(result.stdout.strip())

if original_audio.exists():
    duration = get_duration(original_audio)

    print("Original duration:", round(duration, 2), "seconds")
    print("Original duration:", round(duration / 60, 2), "minutes")
else:
    print("Original audio not found:", original_audio)

Original duration: 591.73 seconds
Original duration: 9.86 minutes


In [3]:
metadata_path = Path("temporary_sections.json")

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, list):
    print("Number of entries:", len(metadata))
    print(json.dumps(metadata[:2], indent=2, ensure_ascii=False))

elif isinstance(metadata, dict):
    print("Top-level keys:", list(metadata.keys()))
    print(
        json.dumps(metadata, indent=2, ensure_ascii=False)[:2000]
    )

Metadata type: list
Number of entries: 10
[
  {
    "filename": "section_000.wav",
    "start": 0,
    "end": 59.9
  },
  {
    "filename": "section_001.wav",
    "start": 58.9,
    "end": 122.55
  }
]


In [4]:
import re
import sys
import json
from pathlib import Path
from rapidfuzz import fuzz

sys.path.insert(0, "../../scripts")
from segments import join_segments

responses_dir = Path("../../data/test/raw_api_responses")

book = Path("../../data/test/book.txt").read_text(
    encoding="utf-8"
)

def normalize(text):
    text = text.lower()

    for mark in ["‘", "’", "ʻ", "ʼ", "`"]:
        text = text.replace(mark, "'")

    text = re.sub(r"[^\w\s']", " ", text)
    return " ".join(text.split())


book_words = normalize(book).split()

response_files = sorted(
    responses_dir.glob("section_*.response")
)

matches = []
previous_start = 0
previous_end = 0

for index, response_path in enumerate(response_files):

    data = json.loads(
        response_path.read_text(encoding="utf-8")
    )

    transcription = join_segments(data["segments"], book_words)

    spoken_text = normalize(transcription)
    word_count = len(spoken_text.split())

    if word_count == 0:
        print("Empty transcription:", response_path.name)
        continue

    window_sizes = [
        max(1, int(word_count * factor))
        for factor in [0.8, 1.0, 1.2]
    ]

    if index == 0:
        search_start = 0
        search_end = len(book_words) - 1

    else:
        search_start = max(0, previous_start - 30)
        search_end = min(
            len(book_words) - 1,
            previous_end + word_count * 3
        )

    best_score = -1
    best_start = 0
    best_end = 0

    step = max(1, word_count // 10)

    for start in range(search_start, search_end + 1, step):

        for size in window_sizes:

            end = min(start + size, len(book_words))

            candidate = " ".join(book_words[start:end])

            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    for start in range(
        max(search_start, best_start - step),
        min(search_end, best_start + step) + 1
    ):
        for size in window_sizes:

            end = min(start + size, len(book_words))
            candidate = " ".join(book_words[start:end])
            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    previous_start = best_start
    previous_end = best_end

    matched_text = " ".join(
        book_words[best_start:best_end]
    )

    matches.append({
        "section": response_path.stem,
        "start_word": best_start,
        "end_word": best_end,
        "similarity": round(best_score, 2),
        "transcription": transcription,
        "book_text": matched_text,
        "needs_review": best_score < 85
    })

    print(
        response_path.stem,
        "| Score:", round(best_score, 2),
        "| Words:", best_start, "-", best_end
    )

output_path = Path("../../data/test/book_matches_draft.json")

output_path.write_text(
    json.dumps(matches, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nSaved:", output_path)

section_000 | Score: 82.35 | Words: 0 - 51
section_001 | Score: 99.39 | Words: 51 - 150
section_002 | Score: 95.03 | Words: 150 - 237
section_003 | Score: 98.26 | Words: 240 - 330
section_004 | Score: 96.3 | Words: 331 - 450
section_005 | Score: 99.57 | Words: 454 - 550
section_006 | Score: 98.76 | Words: 550 - 641
section_007 | Score: 98.91 | Words: 640 - 738
section_008 | Score: 93.84 | Words: 738 - 823
section_009 | Score: 94.55 | Words: 828 - 890

Saved: ../../data/test/book_matches_draft.json


In [6]:

import subprocess

sections = json.loads(
    Path("temporary_sections.json").read_text(encoding="utf-8")
)

sections = sorted(sections, key=lambda s: float(s["start"]))

audio_path = Path("../../data/test/test_audio.mp3")

duration = float(
    subprocess.check_output(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "csv=p=0",
            str(audio_path)
        ],
        text=True
    ).strip()
)

differences = [
    float(b["start"]) - float(a["end"])
    for a, b in zip(sections, sections[1:])
]

print(f"Original duration: {duration:.2f} seconds")
print(f"Total sections: {len(sections)}")
print(f"First section starts: {sections[0]['start']}s")
print(f"Last section ends: {max(float(s['end']) for s in sections):.2f}s")
print(f"Gaps over 0.1s: {sum(d > 0.1 for d in differences)}")
print(f"Overlaps over 0.1s: {sum(d < -0.1 for d in differences)}")


Original duration: 591.73 seconds
Total sections: 10
First section starts: 0s
Last section ends: 591.73s
Gaps over 0.1s: 0
Overlaps over 0.1s: 9


In [15]:
import sys
import json
from pathlib import Path
from difflib import SequenceMatcher

sys.path.insert(0, "../../scripts")
from segments import join_segments

responses_dir = Path("../../data/test/raw_api_responses")
book_words = Path("../../data/test/book.txt").read_text(encoding="utf-8").split()

for response_path in sorted(responses_dir.glob("section_*.response")):
    data = json.loads(response_path.read_text(encoding="utf-8"))

    old_text = " ".join(segment["text"].strip() for segment in data["segments"])
    new_text = join_segments(data["segments"], book_words)

    if old_text == new_text:
        print(f"{response_path.stem}: unchanged")
        continue

    print(f"{response_path.stem}: changed")
    old_words, new_words = old_text.split(), new_text.split()
    matcher = SequenceMatcher(None, old_words, new_words, autojunk=False)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            print(f"    before: {' '.join(old_words[i1:i2])!r:30} after: {' '.join(new_words[j1:j2])!r}")

section_000: unchanged
section_001: changed
    before: "qo 'ydiribdi:"                after: "qo'ydiribdi:"
section_002: unchanged
section_003: unchanged
section_004: changed
    before: 'baxtiyor'                     after: ''
    before: "qo 'riqlaydi,"                after: "qo'riqlaydi,"
section_005: changed
    before: "dimog 'i"                     after: "dimog'i"
section_006: changed
    before: 'Baxtiyor'                     after: ''
section_007: unchanged
section_008: unchanged
section_009: unchanged


In [16]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../scripts")
from segments import join_segments
from sections import load_sections

sections = load_sections("temporary_sections.json")
responses_dir = Path("../../data/test/raw_api_responses")
book_words = Path("../../data/test/book.txt").read_text(encoding="utf-8").split()

texts = {}
for section in sections:
    name = Path(section["filename"]).stem
    data = json.loads((responses_dir / f"{name}.response").read_text(encoding="utf-8"))
    texts[name] = join_segments(data["segments"], book_words).split()

for before, after in zip(sections, sections[1:]):
    a = Path(before["filename"]).stem
    b = Path(after["filename"]).stem
    print(f"{a} ends:   ... {' '.join(texts[a][-4:])}")
    print(f"{b} starts: {' '.join(texts[b][:4])} ...")
    print()

for section in sections:
    print(f"{section['filename']}: audio {section['start']:7.2f} - {section['end']:7.2f}"
          f"   owns {section['own_start']:7.2f} - {section['own_end']:7.2f}")

for before, after in zip(sections, sections[1:]):
    assert before["own_end"] == after["own_start"], "gap or overlap between owned ranges"
print("\nOwned ranges fit together with no gaps or overlaps")

section_000 ends:   ... "Olampanoh, sizga maslahatim bor.
section_001 starts: Chinmochin mamlakatida bir daraxt ...

section_001 ends:   ... kelmoq kerak ekan. O'sha
section_002 starts: O'sha daraxtdan barg keltirib ...

section_002 ends:   ... Baxtiyor yo'l yuribdi, yo'l
section_003 starts: yuribdi, yo'l yursa ham ...

section_003 ends:   ... kimning jigargo'shasisan?" deb so'rabdi.
section_004 starts: Men Xuroson podshosining qizi ...

section_004 ends:   ... gazli dudama qilich beribdi.
section_005 starts: Yigit Gulnoradan birga ketishni ...

section_005 ends:   ... haybat bilan baqirib yaqinlashibdi.
section_006 starts: Baxtiyor ham tavakkal qilib ...

section_006 ends:   ... shahar tinch, osoyishta ekan.
section_007 starts: Baxtiyor otda ko'chadan o'tib ...

section_007 ends:   ... etibsan, balli, debdi. So'ng
section_008 starts: So'ng Baxtiyorni daraxti baxtning ...

section_008 ends:   ... bilan podshoga beribdi, podsho:
section_009 starts: Podsho xursand bo'lib, baxtiyorga ...
